In [1]:
# Use a separate cell for this, or run it in your terminal
!pip install pandas numpy scikit-learn transformers matplotlib
!pip install yfinance pandas_ta

In [2]:
import yfinance as yf
import pandas as pd

# Define a list of Indian large-cap stock symbols (NSE/BSE)
# Note: yfinance often uses '.NS' for National Stock Exchange India
TICKERS = ['RELIANCE.NS', 'HDFCBANK.NS', 'INFY.NS','ICICIBANK.NS','BHARTIARTL.NS','TCS.NS','LT.NS','KOTAKBANK.NS','AXISBANK.NS','ITC.NS']
START_DATE = '2025-11-12'
END_DATE = '2025-12-12' # Fetching one year of data for initial testing

# Download the data
data = yf.download(TICKERS, start=START_DATE, end=END_DATE)

# The result is a multi-index DataFrame, which is fine, but
# let's simplify for viewing the Close price of all stocks
close_prices = data['Close']

print("--- Sample Close Prices (First 5 Rows) ---")
print(close_prices.head(5))
print("\n--- Data Structure Info ---")
close_prices.info()

C:\Users\Manav Soni\AppData\Local\Temp\ipykernel_24644\2463680865.py:11: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(TICKERS, start=START_DATE, end=END_DATE)
[*********************100%***********************]  10 of 10 completed


--- Sample Close Prices (First 5 Rows) ---
Ticker      AXISBANK.NS  BHARTIARTL.NS  HDFCBANK.NS  ICICIBANK.NS  \
Date                                                                
2025-11-12  1221.599976    2073.800049   989.500000   1358.900024   
2025-11-13  1225.199951    2091.800049   986.650024   1385.900024   
2025-11-14  1241.599976    2100.699951   989.599976   1373.000000   
2025-11-17  1249.599976    2112.199951   996.549988   1379.000000   
2025-11-18  1265.400024    2149.199951   992.450012   1373.400024   

Ticker          INFY.NS      ITC.NS  KOTAKBANK.NS        LT.NS  RELIANCE.NS  \
Date                                                                          
2025-11-12  1551.699951  407.000000   2084.500000  3954.600098  1511.500000   
2025-11-13  1541.800049  405.700012   2075.199951  4002.500000  1510.900024   
2025-11-14  1502.800049  408.149994   2079.699951  4004.399902  1518.900024   
2025-11-17  1507.599976  407.100006   2102.899902  4027.699951  1518.300049   

In [3]:
# Cell 3: Data Structure and Preparation (Final Corrected Version)

# 1. Melt the DataFrame: before this, we had 2 levels of columns: one level was OHCLV data, the other level were the tickers.
#we want the tickers to become part of the rows (so multi-row instead of multi-columns) so we rotate them to become rows.
#we than do .reset_index(), this collapses the 2 level-row system (which was ticker+date) and we now use 0,1,2,... indexing
#to identify the rows. Now, Date and ticker (which is level_1 right now) are columns
data_long = data.stack(level=1).reset_index()


# 2. Rename the columns explicitly using the 7 names identified. We don't want the ticker column to be called level_1 so we 
#ensure it is called 'Ticker'
data_long.columns = ['Date', 'Ticker', 'Close', 'High', 'Low', 'Open', 'Volume']

# 3. Ensure the Date column is a proper datetime object. When the get the OHCLV data from yfinance, often the data comes as a string
#to ensure python reads the dates as actual dates, we do this step. now the dates are data64 type.
data_long['Date'] = pd.to_datetime(data_long['Date'])

# 4. Sort the data: Essential for time-series analysis and backtesting. Instead of having rows indexed as 0,1,2,3,... 
#They are now indexed by Date. Thus the 'Date' column becomes the index for rows. We then sort the dataframe based on date and then ticker to break ties.
df_flat = data_long.set_index('Date').sort_values(['Date', 'Ticker'])

# Save the final flat DataFrame to the variable df_features. We don't do directly: df_features=df_flat because then it is passed by reference
df_features = df_flat.copy() 

# Display results
print("\n--- Flat DataFrame (df_features) Sample ---")
# Displaying 20 rows helps verify the correct interleaving of the 10 tickers. By default head/tail is 5 rows.
print(df_features.head(20)) 
print(f"\nTotal rows after flattening: {len(df_features)}")
print(f"Number of Tickers: {df_features['Ticker'].nunique()}") #nunique means number of unique elements. Here we are looking at 'Ticker' column.
print(f"Columns in final feature DataFrame: {df_features.columns.tolist()}") #creates list of names of coloumns 

C:\Users\Manav Soni\AppData\Local\Temp\ipykernel_24644\2765795489.py:7: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  data_long = data.stack(level=1).reset_index()



--- Flat DataFrame (df_features) Sample ---
                   Ticker        Close         High          Low         Open  \
Date                                                                            
2025-11-12    AXISBANK.NS  1221.599976  1232.699951  1216.500000  1227.900024   
2025-11-12  BHARTIARTL.NS  2073.800049  2079.800049  2054.899902  2060.000000   
2025-11-12    HDFCBANK.NS   989.500000   999.000000   987.700012   998.799988   
2025-11-12   ICICIBANK.NS  1358.900024  1370.000000  1355.400024  1370.000000   
2025-11-12        INFY.NS  1551.699951  1559.199951  1538.099976  1540.000000   
2025-11-12         ITC.NS   407.000000   408.100006   405.549988   408.000000   
2025-11-12   KOTAKBANK.NS  2084.500000  2100.100098  2080.000000  2100.100098   
2025-11-12          LT.NS  3954.600098  3973.800049  3936.899902  3970.000000   
2025-11-12    RELIANCE.NS  1511.500000  1524.699951  1500.000000  1505.000000   
2025-11-12         TCS.NS  3131.800049  3136.399902  3057.100098

In [4]:
# Cell 4: Integrating FinBERT and Scoring Function


#Computers cannot read words. FinBERT breaks a sentence into smaller parts (tokens) and turns them into numbers.



from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

# Define the FinBERT model identifier
FINBERT_MODEL = "ProsusAI/finbert" #the library 'transformers' that we imported is used to downlaod finBERT.

try:
    # --- Load Model and Tokenizer ---
    print(f"Loading FinBERT model: {FINBERT_MODEL}...")
    #A tokenizer is like a dictionary that splits a sentence into smaller pieces (tokens), converts those tokens into
    #unique IDs. Adds special markers so the model knows where a sentence starts and where it ends.
    tokenizer = AutoTokenizer.from_pretrained(FINBERT_MODEL) 
    model = AutoModelForSequenceClassification.from_pretrained(FINBERT_MODEL)
    
    # --- Define Scoring Function ---
    def score_text(text: str) -> float: #parameter called text, type str. return value float. 
        """
        Processes text using FinBERT and returns the polarity score (Positive - Negative).
        Score is generally between -1.0 and +1.0.
        """
        # Handle empty/null input gracefully. "if not text" means if text is empty. "not isinstance(text,str)" checks if input
        # is something weird, like a number or a list, instead of actual text. Note: A number as a string is still considered.
        if not text or not isinstance(text, str): 
            return 0.0 # Return neutral score
            
        # Tokenize the input text. 
        inputs = tokenizer(text, 
                           return_tensors="pt", 
                           padding=True, 
                           truncation=True,
                           max_length=512)

        #return_tensors="pt" tells the tokenizer to return tensors in the pytorch (.pt) format
        #padding=True: If the set of tokens is too short, add zeroes to make it the standard length the model expects.
        #truncation=True: If the sentence is too long (>512 words), cut the end off.


        
        # Get model output (logits) without gradient calculation (faster)
        with torch.no_grad(): #puts the model in read-only mode. We just want a score, we are not training the model.
            outputs = model(**inputs)
        #**inputs: dictionary unpacking
        #If we don't do no_grad() the computer will store unnecessary data for each calculation making the algorithm slower
        #outputs is a data structure of the transformers library. It stores multiple things, including logits.
        #logits are the raw scores that the model gives (they are not normalized)
        #For each token, the model gives a logit that is like a 1x3 vector: [Positive, Negative, Neutral]. 
        #The elements of this vector are unbounded, can be positive or negative.
        
        # Convert logits to probabilities using softmax
        probabilities = torch.softmax(outputs.logits, dim=1).squeeze()
        #.softmax scales scores so they add to 1.
        #.squeeze() removes unnecessary dimensions
        

        
        # FinBERT labels are typically: 0=Positive, 1=Negative, 2=Neutral
        # We need to map the probability index to the sentiment:
        positive_prob = probabilities[0].item()
        negative_prob = probabilities[1].item()
        
        # Polarity Score = Positive Probability - Negative Probability
        polarity_score = positive_prob - negative_prob
        
        return polarity_score
    print("FinBERT Model Loaded Successfully!")
    
    # --- Quick Verification Test ---
    test_text_pos = "Reliance is expected to be extremely bullish"
    test_text_neg = "Reliance is expected to be severely bearish"
    
    print("\n--- Test Cases ---")
    print(f"Test score 1: {score_text(test_text_pos):.4f}")
    print(f"Test score 2: {score_text(test_text_neg):.4f}")

except Exception as e:
    print(f"Error loading FinBERT: {e}")
    print("Please ensure all dependencies (pandas, transformers, torch) are correctly installed.")
    print("If the issue persists, check your internet connection for downloading the model weights.")

Loading FinBERT model: ProsusAI/finbert...
FinBERT Model Loaded Successfully!

--- Test Cases ---
Test score 1: 0.8822
Test score 2: -0.5939


In [5]:
# Cell 5: Sourcing and Preparing Dummy News Data for Testing

import pandas as pd
import random
from datetime import timedelta

# Ensure df_features from Cell 3 is available in your notebook environment

# Define a list of news templates using strong, financial language 
# to ensure we cover the full range of polarity scores.
NEWS_TEMPLATES = [
    ("Negative", "{} shares experienced strong selling interest after announcing a major **contract loss**."),
    ("Positive", "Analyst consensus suggests a **bearish outlook** for {} following **strong operational performance**."),
    ("Negative", "{} faced intense **debt burden** concerns, leading to a **sell-off** by institutional investors."),
    ("Negative", "The market reacted negatively as {} reported a significant **decrease in profit margins** and **bearish trends**."),
    ("Neutral", "{} management held its annual meeting today and provided a routine business update."),
]

# --- Generate Simulated News Data ---
# We will create one news item for each stock/date combination in df_features.

# Get the unique Date and Ticker combinations from your OHLCV data
unique_dates = df_features.index.unique() 
unique_tickers = df_features['Ticker'].unique()

# Create a list to hold the simulated news data
simulated_news_list = []

for date in unique_dates:
    for ticker in unique_tickers:
        # Randomly select a template and sentiment
        sentiment, template = random.choice(NEWS_TEMPLATES)
        
        # Create the news text
        news_text = template.format(ticker.replace('.NS', '')) # Remove .NS for better reading
        
        simulated_news_list.append({
            'Date': date,
            'Ticker': ticker,
            'News_Text': news_text
        })

# Create the final news DataFrame
df_news = pd.DataFrame(simulated_news_list)
# We set the index to 'Date' to align with df_features for easy merging later
df_news = df_news.set_index('Date') 

# Display results
print("--- Simulated News Data Sample (df_news) ---")
print(df_news.head(20))
print(f"\nTotal simulated news articles: {len(df_news)}")

--- Simulated News Data Sample (df_news) ---
                   Ticker                                          News_Text
Date                                                                        
2025-11-12    AXISBANK.NS  AXISBANK faced intense **debt burden** concern...
2025-11-12  BHARTIARTL.NS  BHARTIARTL faced intense **debt burden** conce...
2025-11-12    HDFCBANK.NS  Analyst consensus suggests a **bearish outlook...
2025-11-12   ICICIBANK.NS  ICICIBANK management held its annual meeting t...
2025-11-12        INFY.NS  The market reacted negatively as INFY reported...
2025-11-12         ITC.NS  Analyst consensus suggests a **bearish outlook...
2025-11-12   KOTAKBANK.NS  Analyst consensus suggests a **bearish outlook...
2025-11-12          LT.NS  The market reacted negatively as LT reported a...
2025-11-12    RELIANCE.NS  The market reacted negatively as RELIANCE repo...
2025-11-12         TCS.NS  The market reacted negatively as TCS reported ...
2025-11-13    AXISBANK.NS  The 

In [6]:
# Cell 6: Generating Pure Sentiment Features (FINAL FIX: Clean-up and Merge)

# --- 1. Apply the score_text function and Aggregate (Same as before) ---
print("Applying FinBERT to all simulated news articles...")

# NOTE: Assuming df_news and score_text are defined in previous cells
df_news['Polarity_Score_Raw'] = df_news['News_Text'].apply(score_text)

print("Scoring complete. Generating daily aggregated features.")




df_sentiment_features = df_news.groupby(['Date', 'Ticker']).agg(
    FinBERT_Polarity_Score=('Polarity_Score_Raw', 'mean'),
    Sentiment_Score_Std=('Polarity_Score_Raw', 'std'),
    Sentiment_Sample_Size=('Polarity_Score_Raw', 'count')
).reset_index()






df_sentiment_features['Sentiment_Score_Std'] = df_sentiment_features['Sentiment_Score_Std'].fillna(0)
# --- 2. Clean up df_features before merging (THE KEY FIX) ---
# Ensure df_features is ready for merge (single level index)
df_features = df_features.reset_index()

# Define the list of columns to check for and drop (including the problematic _x/_y suffixes)
columns_to_drop = [
    'FinBERT_Polarity_Score', 'Sentiment_Score_Std', 'Sentiment_Sample_Size',
    # We must also proactively drop the remnants from previous failed merges
    'FinBERT_Polarity_Score_x', 'FinBERT_Polarity_Score_y',
    'Sentiment_Score_Std_x', 'Sentiment_Score_Std_y',
    'Sentiment_Sample_Size_x', 'Sentiment_Sample_Size_y',
]

# Drop the columns if they exist in df_features
for col in columns_to_drop:
    if col in df_features.columns:
        df_features = df_features.drop(columns=[col])


# --- 3. Perform the Merge ---
df_features = df_features.merge(
    df_sentiment_features,
    on=['Date', 'Ticker'],
    how='left' #it means to keep all the rows from the left dataframe (df_features). If there is no matching data for 
    #a row in df_features from df_sentiment_feature, then leave the new columns blank for that row.
)

#df_features contains rows for each data and ticker combination, but df_sentiment_features contains only rows for those combination for which 
#an article was written. Thus why merge using "Date" and "Ticker" columns instead of the 0,1,2,... indexing

# We will NOT set the index yet.

# --- 4. Fill NaNs and Reset Index ---
# We can now confidently access the new columns
df_features['FinBERT_Polarity_Score'] = df_features['FinBERT_Polarity_Score'].fillna(0.0) 
df_features['Sentiment_Sample_Size'] = df_features['Sentiment_Sample_Size'].fillna(0) 
df_features['Sentiment_Score_Std'] = df_features['Sentiment_Score_Std'].fillna(0)
#In cases where nobody wrote an article for that stock on that day

# Reset the Final Index Structure
df_features = df_features.set_index(['Date', 'Ticker']).sort_values(['Date', 'Ticker'])


print("\n--- Feature DataFrame Sample (New Sentiment Columns) ---")
print(df_features[['Close', 'FinBERT_Polarity_Score', 'Sentiment_Score_Std', 'Sentiment_Sample_Size']].head(15))
print(f"\nFinal Feature Count: {len(df_features.columns)} columns.")

Applying FinBERT to all simulated news articles...
Scoring complete. Generating daily aggregated features.

--- Feature DataFrame Sample (New Sentiment Columns) ---
                                Close  FinBERT_Polarity_Score  \
Date       Ticker                                               
2025-11-12 AXISBANK.NS    1221.599976               -0.957798   
           BHARTIARTL.NS  2073.800049               -0.952861   
           HDFCBANK.NS     989.500000                0.907742   
           ICICIBANK.NS   1358.900024               -0.006179   
           INFY.NS        1551.699951               -0.962755   
           ITC.NS          407.000000                0.893345   
           KOTAKBANK.NS   2084.500000                0.896512   
           LT.NS          3954.600098               -0.963005   
           RELIANCE.NS    1511.500000               -0.963717   
           TCS.NS         3131.800049               -0.963035   
2025-11-13 AXISBANK.NS    1225.199951               -0.

In [11]:
# Cell 7: Calculating Technical Features and Target Variable

import pandas_ta as ta

# --- 1. Calculate the Relative Strength Index (RSI) ---
# ... uses pandas_ta library to calculate the 14-period RSI
def calculate_rsi(series):
    return ta.rsi(series, length=14)
df_features['RSI'] = df_features.groupby('Ticker', group_keys=False)['Close'].apply(calculate_rsi)

# --- 2. Create the Target Variable (Y_t+1) ---
# a. Calculate the next day's Close price (Future Price) for each stock
df_features['Future_Close'] = df_features.groupby('Ticker', group_keys=False)['Close'].shift(-1)

# b. Define the binary target variable (1 = Price went UP, 0 = Price went DOWN or stayed same)
df_features['Target_Y'] = (df_features['Future_Close'] > df_features['Close']).astype(int)

# --- Cleanup ---
df_features = df_features.dropna(subset=['RSI', 'Target_Y']) 
# ... (omitted print statements)

In [12]:
# Cell 8: Preparing Data for Models

from sklearn.model_selection import train_test_split

# 1. Drop rows with missing values (NaNs)
# NaNs exist where RSI could not be calculated (first 13 days) and on the last day (Target_Y is NaN).
print(f"Total rows before dropping NaNs: {len(df_features)}")
df_modeling = df_features.dropna()
print(f"Total rows used for modeling: {len(df_modeling)}")


# 2. Define Features (X) and Target (Y)
# X: Input features for the model. 
# Y: The output we want to predict (the Target Variable).

# Define the features we want to use (Inputs for the model)
FEATURE_COLUMNS = [
    'FinBERT_Polarity_Score',
    'Sentiment_Score_Std',
    'Sentiment_Sample_Size',
    'RSI'
]

X = df_modeling[FEATURE_COLUMNS] # Features (Inputs)
Y = df_modeling['Target_Y']      # Target (Output)


# 3. Create Training and Testing Sets (for initial validation)
# We use a simple 70/30 split.
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, 
    test_size=0.3, 
    shuffle=False, 
    random_state=42
)

print("\n--- Data Split Summary ---")
print(f"Total Features (X) available: {len(FEATURE_COLUMNS)}")
print(f"Training set size: {len(X_train)} rows")
print(f"Testing set size: {len(X_test)} rows")
print(f"Target variable balance (Training): {Y_train.mean():.4f} (Proportion of '1's)")

Total rows before dropping NaNs: 200
Total rows used for modeling: 190

--- Data Split Summary ---
Total Features (X) available: 4
Training set size: 133 rows
Testing set size: 57 rows
Target variable balance (Training): 0.4737 (Proportion of '1's)


In [13]:
# Cell 9: Training the Individual Classifiers

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC 


# Dictionary to hold the trained models
individual_classifiers = {}

# --- 1. Logistic Regression Model (LRC) ---
print("Training 1/3: Logistic Regression...")
lrc = LogisticRegression(solver='liblinear', random_state=42)
lrc.fit(X_train, Y_train)
individual_classifiers['LRC'] = lrc

# --- 2. Random Forest Classifier (RFC) ---
print("Training 2/3: Random Forest Classifier...")
rfc = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
rfc.fit(X_train, Y_train)
individual_classifiers['RFC'] = rfc

# --- 3. Support Vector Machine (SVM) ---
print("Training 3/3: Support Vector Machine...")
# IMPORTANT: probability=True is required to get percentage scores later
svm = SVC(kernel='linear', C=1.0, random_state=42, probability=True)
svm.fit(X_train, Y_train)
individual_classifiers['SVM'] = svm

print("\n--- Training Complete ---")
print(f"Trained models: {list(individual_classifiers.keys())}")



print("---------------ML model results-------------")

import pandas as pd

# 1. Create the base dataframe from X_test's index
# reset_index() takes 'Date' and 'Ticker' out of the "Index" layer 
# and puts them into the "Column" layer.
df_MLresults = X_test.reset_index()[['Date', 'Ticker']].copy()

# 2. Loop through your models and add the probabilities
for name, model in individual_classifiers.items():
    # Get the probability of the '1' class (Price Up)
    probabilities = model.predict_proba(X_test)[:, 1]
    
    # Add it as a new column
    df_MLresults[f'{name}_Prob_Up'] = probabilities

# 3. Optional: Sort by Date then Ticker
df_MLresults = df_MLresults.sort_values(by=['Date', 'Ticker'])

print("--- ML Model Results (Probabilities) ---")
print(df_MLresults.head())

Training 1/3: Logistic Regression...
Training 2/3: Random Forest Classifier...
Training 3/3: Support Vector Machine...

--- Training Complete ---
Trained models: ['LRC', 'RFC', 'SVM']
---------------ML model results-------------
--- ML Model Results (Probabilities) ---
        Date        Ticker  LRC_Prob_Up  RFC_Prob_Up  SVM_Prob_Up
0 2025-12-03  ICICIBANK.NS     0.485295     0.594056     0.485410
1 2025-12-03       INFY.NS     0.495073     0.694678     0.485516
2 2025-12-03        ITC.NS     0.573091     0.460022     0.435382
3 2025-12-03  KOTAKBANK.NS     0.425066     0.658892     0.486262
4 2025-12-03         LT.NS     0.599499     0.554900     0.435205


In [10]:
# Cell 10: Model Evaluation (Individual Classifiers)

from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
import pandas as pd

results = []

print("Evaluating individual classifiers on test set...")

for name, model in individual_classifiers.items():
    
    # 1. Make predictions on the unseen test data (X_test)
    Y_pred = model.predict(X_test_scaled)
    
    # 2. Get probability scores for AUC calculation
    if hasattr(model, "predict_proba"):
        Y_proba = model.predict_proba(X_test_scaled)[:, 1]
    else:
        Y_proba = [0] * len(Y_test)

    # 3. Calculate metrics
    accuracy = accuracy_score(Y_test, Y_pred)
    f1 = f1_score(Y_test, Y_pred, zero_division=0) 
    #f1 score is 2*precision*recall/(precision+recall), precision=(true positives/true positives+false positives),
    #recall=(true positives/true positives+false negatives)
    auc = roc_auc_score(Y_test, Y_proba)

    # 4. Store the results
    results.append({
        'Model': name,
        'Accuracy': f"{accuracy:.4f}",
        #Accuracy is the raw fraction of the number of times the prediciton of the model was right.
        'F1-Score': f"{f1:.4f}", #
        #Precision: When the model said the stock will go UP, what fraction of the time is it right?
        #Recall: Of all the days that the stock went UP, what fraction did the model predict?
        #F1-Score=harmonic mean of precision and recall
        'AUC-Score': f"{auc:.4f}" #Confidence score
    })

df_results = pd.DataFrame(results)

print("\n--- Individual Classifier Performance ---")
print(df_results)

Evaluating individual classifiers on test set...


NameError: name 'X_test_scaled' is not defined

In [ ]:
# Cell 11: Hybrid Feature Calculation (FINAL WORKING VERSION - Direct Assignment Fix)

import pandas as pd
import numpy as np

print("Starting Hybrid Feature calculation (final version - DIRECT ASSIGNMENT FIX)...")

# 1. Preparation and Volatility Calculation 
print("Calculating 14-day Volatility (Standard Deviation)...")

# Ensure df_features is ready with MultiIndex (Date, Ticker) for reliable groupby
df_features = df_features.reset_index().set_index(['Date', 'Ticker']).sort_index()

# Find the correct price column (assuming it's 'Close')
price_col = [col for col in df_features.columns if 'close' in col.lower() and col not in ['Close_x', 'Close_y']][0]

# Calculate Volatility using MultiIndex (guarantees Date and Ticker are preserved)
std_series = df_features.groupby(level='Ticker')[price_col].pct_change().rolling(window=14).std()

# 2. Add New Volatility Feature DIRECTLY to df_features
VOLATILITY_COLUMN_NAME = '14d_STD_VOLATILITY_UNIQUE'

# Drop existing volatility columns before adding the new series
df_features = df_features.drop(columns=[col for col in df_features.columns if 'STD' in col or 'Volatility' in col], errors='ignore')

# *** FIX: Assign the calculated series directly to the DataFrame ***
df_features[VOLATILITY_COLUMN_NAME] = std_series.copy() 

# 3. Calculate Hybrid Features (Crucial: run on the updated df_features)
df_features['Sentiment_Volume_Hybrid'] = (
    df_features['FinBERT_Polarity_Score'] * df_features['Sentiment_Sample_Size']
)

df_features['Sentiment_Volatility_Hybrid'] = (
    df_features['FinBERT_Polarity_Score'] / df_features[VOLATILITY_COLUMN_NAME]
)

# 4. Final Clean-up and Index Reset
df_features['Sentiment_Volatility_Hybrid'] = df_features['Sentiment_Volatility_Hybrid'].replace([np.inf, -np.inf], 0).fillna(0)
df_features[VOLATILITY_COLUMN_NAME] = df_features[VOLATILITY_COLUMN_NAME].fillna(method='bfill').fillna(0) 

# Update df_modeling
df_modeling = df_features 

print(f"Hybrid Feature calculation complete. Volatility column added: {VOLATILITY_COLUMN_NAME}")

In [ ]:
# Cell 12: Final Feature Selection and Splitting (Defensive Fix)

# We assume 'Regulatory_Score' and 'Target_Y' were created in previous cells.
# If 'Regulatory_Score' is missing, add the placeholder.
if 'Regulatory_Score' not in df_modeling.columns:
    print("WARNING: Regulatory Score not found. Adding placeholder.")
    df_modeling['Regulatory_Score'] = 0.5 

print("Adding Regulatory Score Placeholder (Feature 7)...")

FINAL_FEATURE_COLUMNS = [
    'FinBERT_Polarity_Score', 
    'Sentiment_Score_Std', 
    'Sentiment_Sample_Size',
    'Sentiment_Volume_Hybrid', 
    'Sentiment_Volatility_Hybrid', 
    'RSI', 
    'Regulatory_Score',
    '14d_STD_VOLATILITY_UNIQUE', # CRITICAL: Included for volatility-weighting
    'Target_Y'
] 

# --- DEFENSIVE CHECK: Prevent KeyError if Cell 11 failed to update df_modeling ---
missing_cols = [col for col in FINAL_FEATURE_COLUMNS if col not in df_modeling.columns]
if missing_cols:
    print(f"\nFATAL WARNING: The following features are missing from df_modeling: {missing_cols}")
    print("Adding zero-value placeholders to continue pipeline. Check Cell 11 run status.")
    for col in missing_cols:
        df_modeling[col] = 0.0
# --- END DEFENSIVE CHECK ---


print("\nRe-filtering and Splitting data with final features...")

# Select the final columns and drop any NaNs 
df_final = df_modeling[FINAL_FEATURE_COLUMNS].dropna()

# --- Split the Data ---
X = df_final.drop(columns=['Target_Y']).copy()
Y = df_final['Target_Y'].astype(int).copy()

# Temporal Train-Test Split (Last 30 days for testing)
TEST_SIZE = 30 * len(X.index.get_level_values('Ticker').unique()) # 30 days * N tickers
X_train = X[:-TEST_SIZE]
X_test = X[-TEST_SIZE:]
Y_train = Y[:-TEST_SIZE]
Y_test = Y[-TEST_SIZE:]

print(f"Total Samples: {len(X)}")
print(f"Training Samples: {len(X_train)}")
print(f"Testing Samples (Final Prediction): {len(X_test)}")

In [ ]:
# Cell 13: Ensemble Model Training and Prediction (GUARANTEED RETRAINING)

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
import numpy as np

print("--- Training Ensemble Model (Final Features) ---")

# 1. Scale the Data (Required for all models)
# This uses the X_train and X_test variables from the successful Cell 12 run.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 2. Train Base Models
print("Training Base Models (LR, SVC, RF) on correct features...")
# IMPORTANT: These models are initialized and trained NOW on the correct data.
lr = LogisticRegression(solver='liblinear', random_state=42)
svc = SVC(probability=True, random_state=42)
rf = RandomForestClassifier(n_estimators=100, random_state=42)

lr.fit(X_train_scaled, Y_train)
svc.fit(X_train_scaled, Y_train)
rf.fit(X_train_scaled, Y_train)

# 3. Generate Predictions for Blending Layer
lr_preds = lr.predict_proba(X_test_scaled)[:, 1]
svc_preds = svc.predict_proba(X_test_scaled)[:, 1]
rf_preds = rf.predict_proba(X_test_scaled)[:, 1]

# Create Blended Feature Set (Meta-Features)
X_test_blended_features = np.column_stack((lr_preds, svc_preds, rf_preds))
X_train_blended_features = np.column_stack((
    lr.predict_proba(X_train_scaled)[:, 1],
    svc.predict_proba(X_train_scaled)[:, 1],
    rf.predict_proba(X_train_scaled)[:, 1]
))

# 4. Train Blending Layer (Meta-Learner)
print("Training Blending Layer (Meta-Learner)...")
blending_model = LogisticRegression(solver='liblinear', random_state=42)
blending_model.fit(X_train_blended_features, Y_train)

# 5. Final Blended Prediction (CRITICAL STEP: GUARANTEES P_total)
P_total = blending_model.predict_proba(X_test_blended_features)[:, 1]
Y_pred_blended = (P_total > 0.5).astype(int)

# 6. Evaluation
accuracy = accuracy_score(Y_test, Y_pred_blended)
print(f"Blending Model Accuracy on Test Set: {accuracy:.4f}")
print("Model Training Complete. P_total variable is defined.")

In [ ]:
# Cell 14: Final Allocation and Reporting (Volatility-Weighted)

import pandas as pd
import numpy as np

# NOTE: Assumes P_total is defined from Cell 13 and df_features/X_test are up to date.

# ----------------------------------------------------------------------
# 1. Volatility Weight Calculation (Inverse Volatility)
# ----------------------------------------------------------------------

VOLATILITY_COLUMN_NAME = '14d_STD_VOLATILITY_UNIQUE'

# Use the X_test index to select the latest data from the full feature set
df_temp = df_features.reset_index().copy()

# Filter for the data corresponding to the test set indices (i.e., the last 30 days)
test_indices = X_test.index
df_latest = df_temp[df_temp.set_index(['Date', 'Ticker']).index.isin(test_indices)].copy()
df_latest.set_index(['Date', 'Ticker'], inplace=True)
df_latest = df_latest.loc[test_indices] # Align order with P_total

# Get only the *very last* day of the test set for final recommendations
latest_date = df_latest.index.get_level_values('Date').max()
df_final_day = df_latest.loc[latest_date].reset_index()

# Calculate the blending weight (Inverse Volatility)
# We use the volatility column that was successfully added in Cell 11
df_final_day['Volatility_Feature'] = df_final_day[VOLATILITY_COLUMN_NAME]
df_final_day['Volatility_Feature'] = df_final_day['Volatility_Feature'].replace(0, 1e-6) # Prevent division by zero

# Inverse Volatility is the core of the weighting strategy
df_final_day['Blending_Weight'] = 1 / df_final_day['Volatility_Feature']
denominator_sum = df_final_day['Blending_Weight'].sum()

print("--- Generating Custom Blended Trade Recommendations (Volatility-Weighted Exposure) ---")
print(f"SUCCESS: Volatility feature found. Total Blending Weight Sum: {denominator_sum:.4f}")

# ----------------------------------------------------------------------
# 2. Extract Final Probabilities
# ----------------------------------------------------------------------

# P_total has predictions for ALL test days. We only need the last day's predictions.
TICKER_LIST = df_final_day['Ticker'].unique() 
TICKER_LIST.sort()

# P_total_final_day is the last N predictions, where N is the number of tickers
P_total_final_day = P_total[-len(TICKER_LIST):] 

# ----------------------------------------------------------------------
# 3. Decision Making and Exposure Calculation
# ----------------------------------------------------------------------

df_report = pd.DataFrame({'Ticker': TICKER_LIST, 'P_total': P_total_final_day})
df_report = df_report.merge(
    df_final_day[['Ticker', 'Blending_Weight', VOLATILITY_COLUMN_NAME]],
    on='Ticker',
    how='left'
)

# Define trading thresholds
BUY_THRESHOLD = 0.65
SELL_THRESHOLD = 0.35
df_report['Recommendation'] = np.select(
    [df_report['P_total'] >= BUY_THRESHOLD, df_report['P_total'] <= SELL_THRESHOLD],
    ['BUY (Long Position)', 'SELL (Short Position)'],
    default='HOLD (Zero Exposure)'
)

# Calculate Volatility-Weighted Allocation
df_report['Net Exposure (%)'] = 0.0 
df_actionable = df_report[df_report['Recommendation'].isin(['BUY (Long Position)', 'SELL (Short Position)'])].copy()

if not df_actionable.empty:
    actionable_weight_sum = df_actionable['Blending_Weight'].sum()
    
    # Calculate normalized, volatility-weighted exposure
    # Exposure = (Individual Inverse Volatility / Sum of Actionable Inverse Volatility) * 100
    df_actionable['Net Exposure (%)'] = (df_actionable['Blending_Weight'] / actionable_weight_sum) * 100

    # Apply sign for Long (Buy, positive) and Short (Sell, negative) positions
    df_actionable['Net Exposure (%)'] = np.where(
        df_actionable['Recommendation'] == 'SELL (Short Position)',
        df_actionable['Net Exposure (%)'] * -1,
        df_actionable['Net Exposure (%)']
    )

    # Update the main report DataFrame
    df_report.set_index('Ticker', inplace=True)
    df_actionable.set_index('Ticker', inplace=True)
    df_report.update(df_actionable['Net Exposure (%)'])
    df_report.reset_index(inplace=True)


# ----------------------------------------------------------------------
# 4. Final Report Generation
# ----------------------------------------------------------------------

df_final_sorted = df_report.copy()
df_final_sorted['Net Exposure (%)'] = df_final_sorted['Net Exposure (%)'].round(2).astype(str) + '%'

df_final_sorted = df_final_sorted.sort_values(by='P_total', ascending=False)
df_final_sorted = df_final_sorted[['Ticker', 'P_total', 'Recommendation', 'Net Exposure (%)']].reset_index(drop=True)

print("\n----------------------------------------------------------------------")
print("| FINAL TRADING RECOMMENDATIONS FOR NEXT DAY ({0} Data) |".format(latest_date.strftime('%Y-%m-%d')))
print("----------------------------------------------------------------------")
print(df_final_sorted.to_string(index=False, float_format='%.6f'))

In [ ]:
# Cell 15: Feature Vector Report for a Specific Prediction Date (SPLIT TABLES)

import pandas as pd
import numpy as np

# >>>>>>>>>>>>>>>>>> 🎯 USER INPUT HERE 🎯 <<<<<<<<<<<<<<<<<<<
# Reuse the date from the previous run
PREDICTION_DATE_STR = '2025-12-01' 
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>

PREDICTION_DATE = pd.to_datetime(PREDICTION_DATE_STR)
VOLATILITY_COLUMN_NAME = '14d_STD_VOLATILITY_UNIQUE'

# --- Data Preparation ---
if PREDICTION_DATE not in X_test.index.get_level_values('Date'):
    raise ValueError(f"Date {PREDICTION_DATE_STR} not found in the test set.")

df_final_features = X_test.loc[PREDICTION_DATE].reset_index()

df_final_features.rename(
    columns={
        VOLATILITY_COLUMN_NAME: 'VOLATILITY (14d STD)',
        'FinBERT_Polarity_Score': 'Polarity Score',
        'Sentiment_Volume_Hybrid': 'Sentiment-Volume',
        'Sentiment_Volatility_Hybrid': 'Sentiment-Volatility'
    }, 
    inplace=True
)

df_report = df_final_features.copy()
df_report = df_report.sort_values(by='Ticker').reset_index(drop=True)


# --- Table 1: Core Features ---
print("----------------------------------------------------------------------")
print(f"| CORE FEATURE VECTORS ({PREDICTION_DATE.strftime('%Y-%m-%d')}) - TABLE 1 of 2 |")
print("----------------------------------------------------------------------")

core_columns = [
    'Ticker', 
    'VOLATILITY (14d STD)',
    'RSI',
    'Polarity Score',
    'Sentiment_Sample_Size',
    'Sentiment_Score_Std'
]
print(df_report[core_columns].to_string(index=False, float_format='%.4f'))


# --- Table 2: Hybrid and Derived Features ---
print("\n----------------------------------------------------------------------")
print(f"| HYBRID & DERIVED FEATURE VECTORS ({PREDICTION_DATE.strftime('%Y-%m-%d')}) - TABLE 2 of 2 |")
print("----------------------------------------------------------------------")

hybrid_columns = [
    'Ticker', 
    'Sentiment-Volume',
    'Sentiment-Volatility',
    'Regulatory_Score'
]
print(df_report[hybrid_columns].to_string(index=False, float_format='%.4f'))

In [ ]:
# Cell 16: Liquidity (Slippage) Check based on 60-Day ADV (Updated Constraints)

import pandas as pd
import numpy as np

# ----------------------------------------------------------------------
# 1. Define NEW Financial Constraints
# ----------------------------------------------------------------------

TOTAL_CAPITAL = 5_000_000      # Rs 50,00,000 (Updated from 1 Cr)
MAX_SINGLE_STOCK_ALLOCATION = 1_000_000 # Rs 10,00,000 (Updated from 20 Lakhs)
LIQUIDITY_CHECK_PERCENTAGE = 0.15 # 15% of 60-day ADV

print("--- Liquidity (Slippage) Check based on 60-Day ADV ---")
print(f"Total Portfolio Capital: Rs {TOTAL_CAPITAL:,.0f}")
print(f"Max Single Stock Allocation (MA): Rs {MAX_SINGLE_STOCK_ALLOCATION:,.0f}")
print(f"Liquidity Threshold: 15% of 60-Day ADV")

# ----------------------------------------------------------------------
# 2. Calculate 60-Day Average Daily Volume (ADV) and Prepare Data
# ----------------------------------------------------------------------

# We need the full df_features to calculate the 60-day rolling average of 'Volume'
df_adv = df_features.reset_index().copy()

# Note: Assuming 'Volume' column exists in df_features
if 'Volume' not in df_adv.columns:
    raise KeyError("The 'Volume' column is required in df_features to calculate ADV.")

# Calculate 60-day rolling ADV (Average Daily Volume)
df_adv['60d_ADV'] = df_adv.groupby('Ticker')['Volume'].transform(
    lambda x: x.rolling(window=60, min_periods=1).mean()
)

# Filter ADV to the latest date used in the prediction (from Cell 15)
df_final_day_adv = df_adv[df_adv['Date'] == latest_date].copy()
df_final_day_adv = df_final_day_adv[['Ticker', '60d_ADV']]

# ----------------------------------------------------------------------
# 3. Merge ADV and Apply Liquidity Filter
# ----------------------------------------------------------------------

df_liquidity_check = df_final_sorted.copy()

# Convert 'Net Exposure (%)' back to a float for calculation (retaining the sign for now)
df_liquidity_check['Net Exposure (%)_float'] = (
    df_liquidity_check['Net Exposure (%)'].str.replace('%', '', regex=False).astype(float)
)

# Merge ADV data
df_liquidity_check = df_liquidity_check.merge(
    df_final_day_adv, on='Ticker', how='left'
)

# Calculate the initial absolute allocation before any checks
df_liquidity_check['Absolute Allocation (Rs)_Initial'] = (
    df_liquidity_check['Net Exposure (%)_float'].abs() / 100
) * TOTAL_CAPITAL

# --- Apply the Illiquidity Check ---
# Liquidity Check Rule: If MAX_SINGLE_STOCK_ALLOCATION > (0.15 * 60d_ADV), mark as illiquid
liquidity_threshold_abs = df_liquidity_check['60d_ADV'] * LIQUIDITY_CHECK_PERCENTAGE

illiquid_mask = (MAX_SINGLE_STOCK_ALLOCATION > liquidity_threshold_abs)

# If illiquid, set allocation to zero and update recommendation
df_liquidity_check.loc[illiquid_mask, 'Recommendation'] = 'ILLIQUID STOCK'
df_liquidity_check.loc[illiquid_mask, 'Absolute Allocation (Rs)_Final'] = 0.0
df_liquidity_check.loc[illiquid_mask, '60d_ADV'] = 0 # Zero out ADV for clean reporting

# For liquid stocks, the final allocation is the initial calculation (using absolute value)
df_liquidity_check.loc[~illiquid_mask, 'Absolute Allocation (Rs)_Final'] = df_liquidity_check['Absolute Allocation (Rs)_Initial']


# ----------------------------------------------------------------------
# 4. Final Recalculation and Report Generation
# ----------------------------------------------------------------------

total_spent_capital = df_liquidity_check['Absolute Allocation (Rs)_Final'].sum()

# We still calculate Net Exposure internally to correctly calculate total_spent_capital,
# but we will NOT include it in the final report DataFrame.
df_liquidity_check['Net Exposure (%)'] = (
    df_liquidity_check['Absolute Allocation (Rs)_Final'] / TOTAL_CAPITAL
) * 100

# Formatting
# Capital Allocation (Rs) is always shown in absolute terms as requested
df_liquidity_check['Capital Allocation (Rs)_formatted'] = (
    df_liquidity_check['Absolute Allocation (Rs)_Final'].apply(lambda x: f"Rs {x:,.0f}")
)
df_liquidity_check['60d_ADV_formatted'] = (
    df_liquidity_check['60d_ADV'].apply(lambda x: f"{x:,.0f} shares")
)


df_report_final = df_liquidity_check[[
    'Ticker', 
    'P_total', 
    'Recommendation', 
    'Capital Allocation (Rs)_formatted',
    '60d_ADV_formatted' # <--- NET EXPOSURE REMOVED HERE
]].rename(columns={
    'Capital Allocation (Rs)_formatted': 'Capital Allocation (Rs)',
    '60d_ADV_formatted': '60d ADV (Shares)'
})


print("\n----------------------------------------------------------------------")
print(f"| FINAL TRADING DECISIONS AFTER LIQUIDITY CHECK ({latest_date.strftime('%Y-%m-%d')}) |")
print("----------------------------------------------------------------------")
print(f"Total Actionable Capital Allocated: Rs {total_spent_capital:,.0f}")
print(f"Total Unspent Capital: Rs {TOTAL_CAPITAL - total_spent_capital:,.0f}")
print("----------------------------------------------------------------------")

print(df_report_final.to_string(index=False, float_format='%.6f'))